In [ ]:
import os
import numpy as np
import pandas as pd


class InventoryOptimizer:

    def __init__(self, filepath):

        self.filepath = filepath

        os.makedirs("../outputs/inventory", exist_ok=True)

    ############################################################

    def load_data(self):

        print("=" * 60)
        print("Loading Sales Dataset")
        print("=" * 60)

        self.df = pd.read_csv(

            self.filepath,

            parse_dates=["InvoiceDate"],

            dtype={
                "Invoice": str,
                "StockCode": str
            },

            low_memory=False

        )

        print(self.df.shape)

    ############################################################

    def create_daily_product_sales(self):

        print("\nCreating Product Daily Sales...\n")

        self.df["Date"] = self.df["InvoiceDate"].dt.date

        daily = (

            self.df

            .groupby(

                ["StockCode", "Description", "Date"]

            )

            .agg(

                Daily_Quantity=("Quantity", "sum"),

                Daily_Revenue=("Total_Price", "sum")

            )

            .reset_index()

        )

        self.daily = daily

        print("Daily Product Records :", len(daily))

    ############################################################

    def product_statistics(self):

        print("\nCreating Product Statistics...\n")

        product = (

            self.daily

            .groupby(

                ["StockCode", "Description"]

            )

            .agg(

                Revenue=("Daily_Revenue", "sum"),

                Quantity=("Daily_Quantity", "sum"),

                Avg_Daily_Demand=("Daily_Quantity", "mean"),

                Demand_STD=("Daily_Quantity", "std"),

                Max_Demand=("Daily_Quantity", "max"),

                Min_Demand=("Daily_Quantity", "min"),

                Selling_Days=("Date", "count")

            )

            .reset_index()

        )

        product["Demand_STD"] = product["Demand_STD"].fillna(0)

        self.product = product

        print(product.shape)

    ############################################################

    def save(self):

        self.product.to_csv(

            "../outputs/inventory/product_inventory.csv",

            index=False

        )

        print("\nInventory Dataset Saved")

    ############################################################

    def run(self):

        self.load_data()

        self.create_daily_product_sales()

        self.product_statistics()
        
        self.abc_analysis()

        self.save()

        print("\nInventory Module Step 1 Completed")
        
    ############################################################

def abc_analysis(self):

    print("\nPerforming ABC Analysis...\n")

    # Sort by revenue
    self.product = self.product.sort_values(
        "Revenue",
        ascending=False
    ).reset_index(drop=True)

    # Revenue contribution
    total_revenue = self.product["Revenue"].sum()

    self.product["Revenue_%"] = (
        self.product["Revenue"] / total_revenue
    ) * 100

    # Cumulative %
    self.product["Cumulative_%"] = (
        self.product["Revenue_%"].cumsum()
    )

    # Classification
    def classify(x):

        if x <= 80:
            return "A"

        elif x <= 95:
            return "B"

        else:
            return "C"

    self.product["ABC_Class"] = (
        self.product["Cumulative_%"]
        .apply(classify)
    )

    print()

    print(self.product["ABC_Class"].value_counts())


if __name__ == "__main__":

    obj = InventoryOptimizer(

        r"C:\Users\ka843\Coding\Amdox Internship_project\data\clean\clean_sales.csv"

    )

    obj.run()

Loading Sales Dataset
(1007914, 10)

Creating Product Daily Sales...

Daily Product Records : 534609

Creating Product Statistics...

(5601, 9)

Product Statistics Saved

Inventory Module Step 1 Completed
